# Development Dataset Preparation

In this notebook, I will create a larger chart-and-claim dataset before
making the train, validation and test split.

## Main idea

The first real series contains seven yearly values. I will create
overlapping time windows and three visual versions of every window.

This is a development dataset. It is not yet the final independent
evaluation dataset.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.dataset_builder import (
    build_development_dataset,
)

In [ ]:
source_path = (
    project_root
    / "data"
    / "processed"
    / "eu_unemployment_2019_2025.csv"
)

source_data = pd.read_csv(source_path)
source_data

In [ ]:
chart_output_directory = (
    project_root
    / "data"
    / "generated"
    / "charts"
    / "development"
)

chart_manifest, claim_dataset = (
    build_development_dataset(
        data=source_data,
        output_directory=(
            chart_output_directory
        ),
    )
)

print("Charts:", len(chart_manifest))
print("Examples:", len(claim_dataset))

In [ ]:
chart_manifest.head()

In [ ]:
claim_dataset[
    [
        "example_id",
        "image_path",
        "claim_text",
        "label",
    ]
].head(9)

## Basic checks

The identifiers must be unique, the values must be complete and the
three labels must be balanced.

In [ ]:
print(
    "Unique chart IDs:",
    chart_manifest["chart_id"].is_unique,
)
print(
    "Unique example IDs:",
    claim_dataset["example_id"].is_unique,
)
print(
    "Missing values:",
    claim_dataset.isna().sum().sum(),
)

label_counts = (
    claim_dataset["label"]
    .value_counts()
    .reindex(
        [
            "supported",
            "refuted",
            "not_enough_information",
        ]
    )
)

label_counts

In [ ]:
plt.figure(figsize=(7, 4))

label_counts.plot(kind="bar")

plt.title("Development Examples by Label")
plt.xlabel("Label")
plt.ylabel("Number of examples")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Image check

The following cell displays one generated chart image.

In [ ]:
first_image_path = (
    project_root
    / chart_manifest["image_path"].iloc[0]
)

image = plt.imread(first_image_path)

plt.figure(figsize=(8, 4))
plt.imshow(image)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
chart_manifest_path = (
    project_root
    / "data"
    / "processed"
    / "development_chart_manifest.csv"
)

claim_dataset_path = (
    project_root
    / "data"
    / "processed"
    / "development_chart_claim_dataset.csv"
)

chart_manifest.to_csv(
    chart_manifest_path,
    index=False,
)

claim_dataset.to_csv(
    claim_dataset_path,
    index=False,
)

print("Saved:", chart_manifest_path)
print("Saved:", claim_dataset_path)

## Result

The development dataset contains 30 chart images and 360 chart-and-claim
examples. Each label has 120 examples.

The next step will split the data by `chart_group_id`, so all visual
styles of one time window remain together.